# TinyGPT2 C4 Muon Reproduction Analysis

This notebook analyzes the pulled summary artifacts for
`gpt2_tiny_c4_ppt_reproduction_muon`.

Goals:
1. Compare the analytic fit blocks against the original pre-pretraining seed-derived subspace.
2. Plot downstream pretraining loss curves by initialization, using training tokens on the x-axis.


In [ ]:
from __future__ import annotations

import json
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display
from transformers import AutoConfig, AutoModelForCausalLM

%load_ext autoreload
%autoreload 2


def find_repo_root(start: Path | None = None) -> Path:
    cur = (start or Path.cwd()).resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / ".git").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not locate repo root from current working directory.")


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "notebooks"))
sys.path.insert(0, str(REPO_ROOT / "src" / "platonic_init"))
sys.path.insert(0, str(REPO_ROOT / "src"))

import aesthetics as aes  # noqa: E402

sns = aes.sns

EXPERIMENT_NAME = "gpt2_tiny_c4_ppt_reproduction_muon"
EXPERIMENT_ROOT = REPO_ROOT / "artifacts" / "experiments" / EXPERIMENT_NAME
ANALYSIS_ROOT = EXPERIMENT_ROOT / "analysis"
BASIS_SWEEP_ROOT = ANALYSIS_ROOT / "basis_sweep"
PRETRAIN_ROOT = EXPERIMENT_ROOT / "pretraining"
FIGURE_ROOT = REPO_ROOT / "notebooks" / "figures" / EXPERIMENT_NAME
FIGURE_LAYER_HEATMAP_NAME = "gpt2_tiny_muon_layer_seed_space_error_heatmap"
FIGURE_ATTN_HEATMAP_NAME = "gpt2_tiny_muon_attention_tensor_error_heatmap"
FIGURE_FF_HEATMAP_NAME = "gpt2_tiny_muon_ff_tensor_error_heatmap"
FIGURE_HEATMAP_NAME = "gpt2_tiny_muon_seed_space_tensor_error_heatmap"

CONFIG_PATH = REPO_ROOT / "configs" / "gpt2_tiny_c4_ppt_reproduction_muon.yaml"
config_payload = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
pre_cfg = config_payload["stages"]["prepretrain"]["training"]
TOKENS_PER_STEP = (
    int(pre_cfg["per_device_train_batch_size"])
    * int(pre_cfg.get("gradient_accumulation_steps", 1))
    * int(pre_cfg["block_size"])
)


def load_json(path: Path, default=None):
    if not path.exists():
        return {} if default is None else default
    return json.loads(path.read_text(encoding="utf-8"))


def parse_degree(label: str) -> int | None:
    m = re.search(r"_d(\d+)$", label)
    return int(m.group(1)) if m else None


def init_sort_key(name: str) -> tuple[int, int | str]:
    if name == "random":
        return (0, 0)
    if name == "weight_transfer":
        return (1, 0)
    degree = parse_degree(name)
    if degree is not None:
        return (2, degree)
    return (3, name)


def tensor_sort_key(name: str) -> tuple[int, int, int, str]:
    if name == "transformer.wte.weight":
        return (0, -1, 0, name)
    if name == "transformer.wpe.weight":
        return (1, -1, 0, name)

    layer_match = re.search(r"transformer\.h\.(\d+)\.(.+)", name)
    layer_idx = int(layer_match.group(1)) if layer_match else 10_000
    suffix = layer_match.group(2) if layer_match else name

    component_order = [
        "ln_1",
        "attn.c_attn",
        "attn.c_proj",
        "ln_2",
        "mlp.c_fc",
        "mlp.c_proj",
    ]
    component_rank = len(component_order)
    for idx, prefix in enumerate(component_order):
        if suffix.startswith(prefix):
            component_rank = idx
            break

    weight_bias_rank = (
        0 if suffix.endswith("weight") else 1 if suffix.endswith("bias") else 2
    )
    return (2, layer_idx, component_rank * 10 + weight_bias_rank, name)


def tensor_layer(name: str) -> int | None:
    match = re.search(r"transformer\.h\.(\d+)\.", name)
    return int(match.group(1)) if match else None


def tensor_component_label(name: str) -> str:
    return name.replace("transformer.h.", "")


def is_attention_tensor(name: str) -> bool:
    return ".attn." in name


def is_ff_tensor(name: str) -> bool:
    return ".mlp." in name


def compact_pct_formatter(value: float, _pos=None) -> str:
    pct = value * 100.0
    if pct <= 0:
        return "0%"
    if pct < 0.1:
        return f"{pct:.2f}%"
    if pct < 1:
        return f"{pct:.1f}%"
    if pct < 10:
        return f"{pct:.0f}%"
    return f"{pct:.0f}%"


def human_readable_bytes(num_bytes: int) -> str:
    value = float(num_bytes)
    for unit in ["B", "KB", "MB", "GB"]:
        if value < 1024.0 or unit == "GB":
            return f"{value:.0f} {unit}" if unit != "B" else f"{int(value)} {unit}"
        value /= 1024.0
    return f"{value:.0f} GB"


def format_param_count(num_params: int) -> str:
    value = float(num_params)
    for unit, scale in (("B", 1e9), ("M", 1e6), ("K", 1e3)):
        if value >= scale:
            scaled = value / scale
            if scaled >= 100:
                return f"{scaled:.0f}{unit}"
            if scaled >= 10:
                return f"{scaled:.1f}{unit}"
            return f"{scaled:.2f}{unit}"
    return str(int(value))

## Load Experiment Summaries

This notebook reads only lightweight JSON summaries, so it can run locally without remote checkpoints.


In [ ]:
fit_manifest = load_json(BASIS_SWEEP_ROOT / "fit_blocks.json")
init_eval = load_json(PRETRAIN_ROOT / "init_eval.json", default={"results": []})
init_eval_curves = load_json(
    PRETRAIN_ROOT / "init_eval_basis_curves.json", default={"results": []}
)

fit_rows = []
for report_path in sorted(BASIS_SWEEP_ROOT.glob("analytic_fit_report_*.json")):
    slug = report_path.stem.replace("analytic_fit_report_", "")
    report = load_json(report_path)
    meta = fit_manifest.get(slug, {})
    fit_rows.append(
        {
            "fit_name": slug,
            "basis_type": meta.get("basis_type", "unknown"),
            "degree": parse_degree(slug),
            "mean_relative_error": report.get("mean_relative_error"),
            "num_tensors": len(report.get("tensors", {})),
            "report": report,
        }
    )
fit_df = pd.DataFrame(fit_rows)
if fit_df.empty:
    fit_df = pd.DataFrame(
        columns=[
            "fit_name",
            "basis_type",
            "degree",
            "mean_relative_error",
            "num_tensors",
            "report",
        ]
    )
else:
    fit_df = fit_df.sort_values(["degree", "fit_name"], na_position="last").reset_index(
        drop=True
    )
fit_name_order = fit_df.get("fit_name", pd.Series(dtype=str)).tolist()
fit_display_order = [aes.format_initialization_label(name) for name in fit_name_order]
if "fit_name" in fit_df:
    fit_df["fit_name"] = pd.Categorical(
        fit_df["fit_name"], categories=fit_name_order, ordered=True
    )
if "degree" in fit_df:
    fit_df["degree_label"] = fit_df["degree"].astype("Int64").astype(str)
else:
    fit_df["degree_label"] = pd.Series(dtype=str)
fit_df["display_name"] = pd.Categorical(
    fit_display_order, categories=fit_display_order, ordered=True
)

INIT_PALETTE = aes.initialization_palette(fit_name_order)
INIT_LABEL_ORDER = aes.initialization_label_order(fit_name_order)

seed_rows = []
first_artifact_path = next(
    iter(sorted(BASIS_SWEEP_ROOT.glob("analytic_subspace_*.pt"))), None
)
if first_artifact_path is not None:
    artifact_payload = torch.load(first_artifact_path, map_location="cpu")
    for tensor_name, info in artifact_payload.items():
        if str(tensor_name).startswith("__"):
            continue
        seed_rows.append(
            {
                "tensor": tensor_name,
                "numel": info.get("numel"),
                "evr_1": None,
                "evr_2": None,
                "evr_cum_2": None,
            }
        )
seed_df = pd.DataFrame(seed_rows)
if seed_df.empty:
    seed_df = pd.DataFrame(columns=["tensor", "numel", "evr_1", "evr_2", "evr_cum_2"])
else:
    seed_df = seed_df.sort_values("tensor").reset_index(drop=True)
seed_numel_map = (
    seed_df.set_index("tensor")["numel"]
    if not seed_df.empty
    else pd.Series(dtype=float)
)

summary_results = init_eval.get("results", [])
curve_results = init_eval_curves.get("results") or summary_results

print(f"Experiment root: {EXPERIMENT_ROOT}")
print(f"Tokens per training step: {TOKENS_PER_STEP:,}")
summary_count = len(summary_results)
print(f"Found {len(fit_df)} fit reports and {summary_count} downstream run summaries.")
display(
    fit_df[
        ["display_name", "basis_type", "degree", "mean_relative_error", "num_tensors"]
    ]
)
display(seed_df.head(10))

## Fit Blocks vs Original Pre-Pretraining Seeds

The analytic fit reports are computed on the PCA components extracted from the original pre-pretraining seeds.
Lower reconstruction error means a fit block better matches the shared seed-derived subspace.


In [ ]:
seed_overview = pd.DataFrame(
    {
        "stat": [
            "num_tensors",
            "total_params_analyzed",
            "metadata_source",
        ],
        "value": [
            len(seed_df),
            int(seed_df["numel"].sum()) if not seed_df.empty else None,
            "analytic_delta_artifact" if not seed_df.empty else None,
        ],
    }
)
display(seed_overview)
display(fit_df[["display_name", "degree", "mean_relative_error", "num_tensors"]])

In [ ]:
if fit_df.empty:
    print("No analytic fit reports found yet; skipping fit-error bar plot.")
else:
    fig = plt.figure(figsize=(aes.PAPER_WIDTH_IN, aes.FIG_HEIGHT_SINGLE_ROW_IN * 1.2))

    with sns.plotting_context("paper", font_scale=1, rc=aes.rcs):
        ax = fig.add_subplot(1, 1, 1)
        sns.lineplot(
            data=fit_df,
            x="degree",
            y="mean_relative_error",
            marker="o",
            color=aes.NONSEMANTIC_COLOR,
            linewidth=2.0,
            markersize=7,
            ax=ax,
        )
        ax.set_title("Analytic Fit Error vs Seed-Derived Subspace")
        ax.set_xlabel("Chebyshev Initialization Degree")
        ax.set_ylabel("Relative error")
        ax.set_xscale("log", base=2)
        ax.set_xticks(fit_df["degree"].tolist())
        ax.xaxis.set_major_formatter(aes.NICE_FORMATTER)
        ax.tick_params(axis="x", rotation=0)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    aes.save_figure(FIGURE_ROOT / "gpt2_tiny_muon_fit_error_vs_degree", fig=fig)
    plt.show()

**Note**
The token/position embedding tensors (`wte`/`wpe`) are excluded from the heatmap below. They are persistently hard to fit, but that is less informative here because embeddings are reset/reprojected before downstream pretraining.


In [ ]:
if fit_df.empty or seed_numel_map.empty:
    print(
        "Fit reports or tensor-size metadata are not available yet; "
        "skipping fit heatmaps."
    )
else:
    tensor_error_rows = []
    for _, row in fit_df.iterrows():
        for tensor_name, info in row["report"].get("tensors", {}).items():
            tensor_error_rows.append(
                {
                    "fit_name": row["fit_name"],
                    "degree": row["degree"],
                    "tensor": tensor_name,
                    "layer": tensor_layer(tensor_name),
                    "component": tensor_component_label(tensor_name),
                    "mean_component_relative_error": info.get(
                        "mean_component_relative_error"
                    ),
                    "numel": seed_numel_map.get(tensor_name),
                }
            )

    tensor_error_df = pd.DataFrame(tensor_error_rows)
    non_embedding_tensor_error_df = tensor_error_df[
        ~tensor_error_df["tensor"].str.contains(r"wte|wpe", regex=True)
    ].copy()
    degree_lookup = fit_df.set_index("fit_name")["degree"].to_dict()

    layer_error_df = (
        non_embedding_tensor_error_df.dropna(subset=["layer", "numel"])
        .groupby(["layer", "fit_name"], as_index=False)
        .apply(
            lambda frame: pd.Series(
                {
                    "weighted_error": np.average(
                        frame["mean_component_relative_error"],
                        weights=frame["numel"],
                    )
                }
            ),
            include_groups=False,
        )
    )
    layer_heatmap_df = (
        layer_error_df.pivot(index="layer", columns="fit_name", values="weighted_error")
        .reindex(columns=fit_name_order)
        .sort_index(ascending=False)
    )
    layer_heatmap_df.index = [str(int(layer)) for layer in layer_heatmap_df.index]
    layer_heatmap_df.columns = [
        str(int(degree_lookup[name])) for name in layer_heatmap_df.columns
    ]
    layer_tick_labels = [
        label if idx % 2 == 0 else ""
        for idx, label in enumerate(layer_heatmap_df.index.tolist())
    ]

    fig = plt.figure(
        figsize=(aes.PAPER_WIDTH_IN, max(1.8, 0.24 * len(layer_heatmap_df)))
    )
    with sns.plotting_context("paper", font_scale=0.85, rc=aes.rcs):
        ax = fig.add_subplot(1, 1, 1)
        sns.heatmap(
            layer_heatmap_df, cmap="magma_r", ax=ax, yticklabels=layer_tick_labels
        )
        ax.set_title("Layer-Aggregated Seed-Space Error")
        ax.set_xlabel("Chebyshev Initialization Degree")
        ax.set_ylabel("Layer")
        ax.tick_params(axis="x", rotation=0)
        ax.tick_params(axis="y", length=0)

    aes.save_figure(FIGURE_ROOT / FIGURE_LAYER_HEATMAP_NAME, fig=fig)
    plt.show()

    def compact_component_label(name: str) -> str:
        mapping = {
            "attn.c_attn.weight": "QKV W",
            "attn.c_attn.bias": "QKV b",
            "attn.c_proj.weight": "Out W",
            "attn.c_proj.bias": "Out b",
            "mlp.c_fc.weight": "FC in W",
            "mlp.c_fc.bias": "FC in b",
            "mlp.c_proj.weight": "FC out W",
            "mlp.c_proj.bias": "FC out b",
        }
        layer_match = re.search(r"transformer\.h\.\d+\.(.+)", name)
        suffix = layer_match.group(1) if layer_match else name
        return mapping.get(suffix, suffix.replace(".", " "))

    def component_heatmap_bundle(filter_fn):
        component_df = non_embedding_tensor_error_df[
            non_embedding_tensor_error_df["tensor"].map(filter_fn)
        ].copy()
        component_df = component_df.dropna(subset=["layer"])
        component_meta = (
            component_df[["layer", "tensor"]]
            .drop_duplicates()
            .assign(
                sort_key=lambda frame: frame["tensor"].map(tensor_sort_key),
                short_label=lambda frame: frame["tensor"].map(compact_component_label),
            )
            .sort_values("sort_key", ascending=False)
        )
        base_heatmap_df = component_df.pivot(
            index="tensor", columns="fit_name", values="mean_component_relative_error"
        ).reindex(index=component_meta["tensor"].tolist(), columns=fit_name_order)

        display_rows = []
        display_labels = []
        layer_centers = []
        separator_positions = []
        row_position = 0

        ordered_layers = component_meta["layer"].drop_duplicates().tolist()
        for layer_idx, layer in enumerate(ordered_layers):
            layer_meta = component_meta[component_meta["layer"] == layer]
            layer_start = row_position
            for _, meta_row in layer_meta.iterrows():
                display_rows.append(base_heatmap_df.loc[meta_row["tensor"]])
                display_labels.append(meta_row["short_label"])
                row_position += 1
            layer_centers.append((int(layer), layer_start + (len(layer_meta) / 2.0)))
            if layer_idx < len(ordered_layers) - 1:
                display_rows.append(pd.Series(np.nan, index=base_heatmap_df.columns))
                display_labels.append("")
                row_position += 1
                separator_positions.append(row_position)

        heatmap_df = pd.DataFrame(display_rows, columns=base_heatmap_df.columns)
        heatmap_df.columns = [
            str(int(degree_lookup[name])) for name in heatmap_df.columns
        ]
        return heatmap_df, display_labels, layer_centers, separator_positions

    def plot_component_heatmap(
        heatmap_df,
        display_labels,
        layer_centers,
        separator_positions,
        *,
        title,
        figure_name,
    ):
        fig = plt.figure(figsize=(aes.PAPER_WIDTH_IN, max(3.2, 0.28 * len(heatmap_df))))
        with sns.plotting_context("paper", font_scale=0.8, rc=aes.rcs):
            ax = fig.add_subplot(1, 1, 1)
            sns.heatmap(
                heatmap_df,
                cmap="magma_r",
                ax=ax,
                yticklabels=display_labels,
                cbar_kws={"shrink": 0.8},
            )
            for pos in separator_positions:
                ax.axhline(pos, color="white", linewidth=6)
            for layer, center in layer_centers:
                ax.text(
                    -0.55,
                    center,
                    str(layer),
                    rotation=0,
                    va="center",
                    ha="right",
                    fontsize=plt.rcParams["font.size"] * 0.85,
                    transform=ax.transData,
                    clip_on=False,
                )
            ax.set_title(title)
            ax.set_xlabel("Chebyshev Initialization Degree")
            ax.set_ylabel("Component")
            ax.tick_params(axis="x", rotation=0)
            ax.tick_params(axis="y", length=0)
            fig.subplots_adjust(left=0.26)

        aes.save_figure(FIGURE_ROOT / figure_name, fig=fig)
        plt.show()

    attn_heatmap_df, attn_labels, attn_layer_centers, attn_separator_positions = (
        component_heatmap_bundle(is_attention_tensor)
    )
    plot_component_heatmap(
        attn_heatmap_df,
        attn_labels,
        attn_layer_centers,
        attn_separator_positions,
        title="Attention Tensor Error by Layer",
        figure_name=FIGURE_ATTN_HEATMAP_NAME,
    )

    ff_heatmap_df, ff_labels, ff_layer_centers, ff_separator_positions = (
        component_heatmap_bundle(is_ff_tensor)
    )
    plot_component_heatmap(
        ff_heatmap_df,
        ff_labels,
        ff_layer_centers,
        ff_separator_positions,
        title="Feed-Forward Tensor Error by Layer",
        figure_name=FIGURE_FF_HEATMAP_NAME,
    )

## Compact Artifact Sizes

The learned initialization is now stored as a compact **analytic delta** artifact under `analysis/basis_sweep/analytic_subspace_*.pt`. These files contain the per-tensor Chebyshev coefficients and minimal reconstruction metadata needed to apply the learned update to a fresh random initialization; they do not store dense merged weights.

In [ ]:
artifact_rows = []
for path in sorted(BASIS_SWEEP_ROOT.glob("analytic_subspace_*.pt")):
    artifact_rows.append(
        {
            "artifact": path.name,
            "degree": parse_degree(path.stem),
            "size_bytes": path.stat().st_size,
            "size_kib": path.stat().st_size / 1024.0,
            "size_mib": path.stat().st_size / (1024.0**2),
        }
    )

artifact_df = pd.DataFrame(artifact_rows)
model_name = pre_cfg["model_name_or_path"]
model_cfg = AutoConfig.from_pretrained(model_name)
model_for_size = AutoModelForCausalLM.from_config(model_cfg)
model_num_params = sum(param.numel() for param in model_for_size.parameters())
del model_for_size
model_fp32_bytes = model_num_params * 4
model_fp32_mib = model_fp32_bytes / (1024.0**2)
model_comparison_df = pd.DataFrame(
    {
        "label": [
            "Stored init artifacts",
            "Initialized model (fp16/bf16)",
            "Initialized model (fp32)",
        ],
        "size_bytes": [
            artifact_df["size_bytes"].max() if not artifact_df.empty else None,
            model_fp32_bytes / 2,
            model_fp32_bytes,
        ],
        "size_mib": [
            artifact_df["size_mib"].max() if not artifact_df.empty else None,
            model_fp32_mib / 2,
            model_fp32_mib,
        ],
    }
)

if artifact_df.empty:
    print("No compact analytic delta artifacts found yet.")
else:
    artifact_df = artifact_df.sort_values("degree").reset_index(drop=True)
    artifact_df["frac_of_model_fp32"] = artifact_df["size_bytes"] / model_fp32_bytes
    artifact_df["compression_vs_model_fp32"] = (
        model_fp32_bytes / artifact_df["size_bytes"]
    )
    display(artifact_df)
    display(model_comparison_df)
    print(
        f"Smallest artifact: {artifact_df['size_kib'].min():.1f} KiB | "
        f"Largest artifact: {artifact_df['size_kib'].max():.1f} KiB"
    )
    print(
        f"Largest artifact is {artifact_df['frac_of_model_fp32'].max():.4%} of "
        "an fp32 initialized model."
    )

    fig = plt.figure(figsize=(aes.PAPER_WIDTH_IN, aes.FIG_HEIGHT_SINGLE_ROW_IN * 1.4))
    with sns.plotting_context("paper", font_scale=1, rc=aes.rcs):
        ax = fig.add_subplot(1, 1, 1)
        sns.lineplot(
            data=artifact_df,
            x="degree",
            y="frac_of_model_fp32",
            marker="o",
            color=aes.NONSEMANTIC_COLOR,
            linewidth=2.0,
            markersize=7,
            ax=ax,
        )
        ax.axhline(
            1.0,
            color=INIT_PALETTE["PPT"],
            linewidth=2.0,
            linestyle="-",
            alpha=1.0,
        )
        ax.annotate(
            human_readable_bytes(int(model_fp32_bytes)),
            (artifact_df["degree"].max(), 1.0),
            xytext=(0, 5),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=7,
            color=INIT_PALETTE["PPT"],
        )
        ax.set_xlabel("Chebyshev Initialization Degree")
        ax.set_ylabel("Relative size")
        ax.set_xscale("log", base=2)
        ax.set_yscale("log")
        ax.set_xticks(artifact_df["degree"].tolist())
        ax.get_xaxis().set_major_formatter(aes.NICE_FORMATTER)
        ax.get_yaxis().set_major_formatter(plt.FuncFormatter(compact_pct_formatter))
        ax.grid(axis="y", which="major", linestyle="--", linewidth=0.8, alpha=0.35)
        for row in artifact_df.itertuples(index=False):
            x_offset = 9 if row.degree == 8 else 0
            ax.annotate(
                human_readable_bytes(int(row.size_bytes)),
                (row.degree, row.frac_of_model_fp32),
                xytext=(x_offset, 6),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=7,
                color=aes.darken(aes.NONSEMANTIC_COLOR, by=0.1),
            )
        ax.tick_params(axis="x", rotation=0)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    aes.set_figure_title(
        fig,
        "Stored Initialization Size vs Model Weights",
        subtitle=(
            f"GPT-2 Tiny ({format_param_count(model_num_params)} params) "
            "relative to fp32 weights"
        ),
        y=1.1,
        subtitle_offset=0.1,
        subtitle_kwargs={"fontsize": plt.rcParams["font.size"] * 0.9},
    )
    aes.save_figure(FIGURE_ROOT / "gpt2_tiny_muon_init_size_comparison", fig=fig)
    plt.show()

## Downstream Pretraining Loss Curves by Initialization

This section plots the available downstream eval-loss curves from `pretraining/init_eval*.json`,
with one series per initialization label and training tokens on the x-axis.


In [ ]:
result_rows = []
for row in summary_results:
    raw_label = row.get("label", row.get("variant", "unknown"))
    result_rows.append(
        {
            "label": aes.format_initialization_label(raw_label),
            "raw_label": raw_label,
            "variant": row.get("variant"),
            "init_mode": row.get("init_mode"),
            "best_eval_loss": row.get("best_eval_loss"),
            "final_eval_loss": row.get("final_eval_loss"),
            "train_loss": row.get("train_loss"),
        }
    )
result_df = pd.DataFrame(result_rows)
if result_df.empty:
    result_df = pd.DataFrame(
        columns=[
            "label",
            "raw_label",
            "variant",
            "init_mode",
            "best_eval_loss",
            "final_eval_loss",
            "train_loss",
        ]
    )
else:
    result_df = result_df.sort_values("best_eval_loss").reset_index(drop=True)
display(result_df)

curve_rows = []
for row in curve_results:
    raw_label = row.get("label", row.get("variant", "unknown"))
    display_label = aes.format_initialization_label(raw_label)
    for point in row.get("eval_curve", []):
        step = point.get("step")
        curve_rows.append(
            {
                "label": display_label,
                "raw_label": raw_label,
                "step": step,
                "tokens": step * TOKENS_PER_STEP if step is not None else None,
                "loss": point.get("eval_loss"),
                "metric": "eval_loss",
                "basis": row.get("basis"),
                "init_mode": row.get("init_mode"),
            }
        )
    for point in row.get("train_curve") or []:
        step = point.get("step")
        curve_rows.append(
            {
                "label": display_label,
                "raw_label": raw_label,
                "step": step,
                "tokens": step * TOKENS_PER_STEP if step is not None else None,
                "loss": point.get("loss"),
                "metric": "train_loss",
                "basis": row.get("basis"),
                "init_mode": row.get("init_mode"),
            }
        )

curves_df = pd.DataFrame(curve_rows)
available_labels = (
    sorted(curves_df.label.unique().tolist()) if not curves_df.empty else []
)
print(f"Available run labels: {available_labels}")
available_metrics = (
    sorted(curves_df.metric.unique().tolist()) if not curves_df.empty else []
)
print(f"Available metrics: {available_metrics}")
display(curves_df.head())

In [ ]:
if curves_df.empty:
    print(
        "No downstream pretraining curves were found in the pulled summaries "
        "yet; skipping loss plot."
    )
    plot_df = pd.DataFrame()
    label_order = []
else:
    metric_to_plot = "eval_loss"
    selected_labels = ["Baseline (Gaussian, σ=0.02)", "PPT", "Plato (d=6)"]
    plot_df = curves_df[curves_df["metric"] == metric_to_plot].copy()
    plot_df = plot_df[plot_df["label"].isin(selected_labels)].copy()
    label_order = [
        label for label in selected_labels if label in plot_df["label"].unique()
    ]

    random_best_loss = None
    random_best_tokens = None
    random_rows = plot_df[plot_df["label"] == "Baseline (Gaussian, σ=0.02)"]
    if not random_rows.empty:
        best_random = random_rows.sort_values(["loss", "tokens"]).iloc[0]
        random_best_loss = float(best_random["loss"])
        random_best_tokens = float(best_random["tokens"])

    threshold_markers = []
    legend_label_map = {label: label for label in label_order}
    if (
        random_best_loss is not None
        and random_best_tokens is not None
        and random_best_tokens > 0
    ):
        for label in label_order:
            if label == "Baseline (Gaussian, σ=0.02)":
                continue
            label_rows = plot_df[plot_df["label"] == label].sort_values("tokens")
            crossings = label_rows[label_rows["loss"] <= random_best_loss]
            if crossings.empty:
                final_loss = float(label_rows.iloc[-1]["loss"])
                gap = final_loss - random_best_loss
                legend_label_map[label] = f"{label} • +{gap:.3f} nats/token"
                continue
            first_crossing = crossings.iloc[0]
            crossing_tokens = float(first_crossing["tokens"])
            crossing_loss = float(random_best_loss)
            crossing_pos = label_rows.index.get_loc(first_crossing.name)
            if crossing_pos > 0:
                prev_row = label_rows.iloc[crossing_pos - 1]
                prev_tokens = float(prev_row["tokens"])
                prev_loss = float(prev_row["loss"])
                next_tokens = float(first_crossing["tokens"])
                next_loss = float(first_crossing["loss"])
                if prev_loss > random_best_loss and next_loss < prev_loss:
                    loss_delta = next_loss - prev_loss
                    if loss_delta != 0:
                        frac = (random_best_loss - prev_loss) / loss_delta
                        crossing_tokens = prev_tokens + frac * (
                            next_tokens - prev_tokens
                        )
            savings_pct = 100.0 * (1.0 - crossing_tokens / random_best_tokens)
            legend_label_map[label] = f"{label} • {savings_pct:.0f}% savings"
            threshold_markers.append(
                {
                    "label": label,
                    "tokens": crossing_tokens,
                    "loss": crossing_loss,
                }
            )

    fig = plt.figure(figsize=(aes.PAPER_WIDTH_IN, aes.FIG_HEIGHT_DOUBLE_ROW_IN))
    with sns.plotting_context("paper", font_scale=1, rc=aes.rcs):
        ax = fig.add_subplot(1, 1, 1)
        sns.lineplot(
            data=plot_df,
            x="tokens",
            y="loss",
            hue="label",
            hue_order=label_order,
            palette=INIT_PALETTE,
            linewidth=2.0,
            ax=ax,
        )
        if threshold_markers:
            marker_df = pd.DataFrame(threshold_markers)
            sns.scatterplot(
                data=marker_df,
                x="tokens",
                y="loss",
                hue="label",
                hue_order=label_order,
                palette=INIT_PALETTE,
                s=80,
                legend=False,
                ax=ax,
                zorder=5,
            )
        if random_best_loss is not None:
            ax.axhline(
                random_best_loss,
                color=aes.darken("#999999", by=0.15),
                linewidth=1.0,
                linestyle="--",
            )
        ax.set_title("Loss by Initialization (GPT-2 Tiny)")
        ax.set_xlabel("Training tokens")
        ax.xaxis.set_major_formatter(aes.NICE_FORMATTER)
        ax.set_ylabel("Validation loss (C4)")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        legend = ax.legend(loc="upper right", frameon=False, title=None)
        if legend is not None:
            for text in legend.texts:
                raw = text.get_text()
                text.set_text(legend_label_map.get(raw, raw))

    aes.save_figure(
        FIGURE_ROOT / "gpt2_tiny_muon_downstream_eval_loss_selected_initializations",
        fig=fig,
    )
    plt.show()

## Notes

- The fit-quality section is tied directly to the original pre-pretraining seeds because the analytic fit reports are computed on the seed-derived PCA subspace.
- The downstream section plots whatever run summaries are currently present on disk. If only `random` and `weight_transfer` are available so far, the notebook will reflect that.
- Once fit-based downstream runs finish and their summaries are synced, rerunning this notebook should extend the curve plot automatically.
